# NB4 — LOSO: Janela Ótima, Nível Ótimo e Comparação de Modelos

**Pipeline de Predição de Crises Epilépticas a partir de EEG**

## Fluxo

```
NB3 → data/features/{dataset}__{pat}.npz   (X_pre, X_inter, t_pre, t_inter, ctx_ids)
NB2 → data/preictal_estimate.json          (PRE_SEC individual por paciente)
                         ↓
NB4 → Etapa 1: qual janela W performa melhor?       (livre=W, fixo=nível máx, modelo=RF)
       Etapa 2: qual nível de canais performa melhor? (livre=nível, fixo=W consenso, modelo=RF)
       Etapa 3: qual modelo performa melhor?          (livre=modelo, fixo=W+nível ótimos)
```

## Design experimental — LOSO por contexto de crise

Cada `ctx_id` é uma unidade de leave-one-out: treina em todos os outros contextos
do paciente (undersample 1:1 pré/inter no treino) e testa no contexto deixado de
fora **sem undersampling**.

## Decisões metodológicas

| Decisão | Escolha | Razão |
|---------|---------|-------|
| Modelo principal | Random Forest | Literatura de predição ictal |
| Outros modelos | XGB, SVM, LR, NB, kNN, MDC | Comparação secundária |
| Undersample | 1:1 pré/inter, 5 seeds | Estabilidade |
| Z-score | Por contexto antes de empilhar | Remove drift inter-sessão |
| FP/h (Opção B) | Interictal completo do teste | Clinicamente válido entre janelas |
| SelectKBest | Desativado | Prejudicava RF com poucos dados por fold |

## Correções aplicadas

| # | Bug | Correção |
|---|-----|----------|
| 1 | `all_contexts` usava cp∪ci → folds com Xi_te=[] | UNIÃO das interseções cp∩ci |
| 2 | `return None` quando fold vazio → N_Ctx subreportado | Dict com nan + campo erro |
| 3 | Pacientes com 2 ctx excluídos silenciosamente | Fallback com W_DEFAULT_S |
| 4 | N_Ctx_Total calculado dos folds | Carregado do pipeline_manifest.json |

## Limitações declaradas

- W5 estimado sobre o mesmo sinal do experimento — interpretar com cautela
- SeizeIT2 fixado em R0 por limitação de hardware (2 canais nativos)
- Undersample 1:1 no treino; distribuição real (desbalanceada) no teste
- N_SEEDS=5 para estabilidade do undersample


In [1]:
import subprocess, sys
for pkg in ['xgboost', 'scikit-learn', 'tqdm']:
    try: __import__(pkg.replace('scikit-learn','sklearn'))
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Dependências OK.')

Dependências OK.


## 1. Imports e configuração

In [ ]:
import json, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from xgboost import XGBClassifier

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    def tqdm(it, **kw): return it

warnings.filterwarnings('ignore')

ROOT     = Path('data')
FEAT_DIR = ROOT / 'features'
PRE_EST  = ROOT / 'preictal_estimate.json'
MANIFEST = ROOT / 'pipeline_manifest.json'
OUT_DIR  = ROOT / 'results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_FEATS_CH = 19

# Janelas experimentais
WINDOWS_FIXED = {
    'W1': 10 * 60,
    'W2': 15 * 60,
    'W3': 20 * 60,
    'W4': 25 * 60,
}
# W5 = PRE_SEC individual; fallback se não disponível
PRESEC_FALLBACK_S = 10.2 * 60   # mediana global — atualizar após novo NB2

# Feature selection
N_FEATURES_SELECT = None  # desativado — prejudica RF com poucos dados

# Treino
ZSCORE_BY_CTX = True
N_SEEDS       = 5
SEEDS         = [42, 43, 44, 45, 46]
W_DEFAULT_S   = 25 * 60   # fallback para pacientes com 2 contextos (W4)

print('Configuração carregada.')
print(f'  Janelas: { {k: v//60 for k,v in WINDOWS_FIXED.items()} } min + W5')
print(f'  PRESEC_FALLBACK_S = {PRESEC_FALLBACK_S/60:.1f}min')
print(f'  ZSCORE_BY_CTX={ZSCORE_BY_CTX}  N_SEEDS={N_SEEDS}  W_DEFAULT={W_DEFAULT_S//60}min')

Configuração carregada.
  Janelas: {'W1': 10, 'W2': 15, 'W3': 20, 'W4': 25} min + W5
  PRESEC_FALLBACK_S = 10.2min
  ZSCORE_BY_CTX=True  N_SEEDS=5  W_DEFAULT=25min


c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Classificador de Distância Mínima (MDC)

In [3]:
class MinimumDistanceClassifier:
    """Distância mínima ao centróide de classe. Interface sklearn."""
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.centroids_ = {c: X[y == c].mean(axis=0) for c in self.classes_}
        return self
    def _dists(self, X):
        return np.column_stack([np.linalg.norm(X - self.centroids_[c], axis=1)
                                 for c in self.classes_])
    def predict(self, X):
        return self.classes_[np.argmin(self._dists(X), axis=1)]
    def predict_proba(self, X):
        d = self._dists(X)
        inv = 1.0 / (d + 1e-10)
        p = inv / inv.sum(axis=1, keepdims=True)
        return p[:, np.argsort(self.classes_)]

print('MinimumDistanceClassifier definido.')

MinimumDistanceClassifier definido.


## 3. Fábricas de modelos

In [4]:
def make_rf():
    # Validado empiricamente em chb06 (Config A vs B):
    # remoção do min_samples_leaf deu +0.067 AUC
    return RandomForestClassifier(n_estimators=200, max_depth=12,
                                  random_state=42, n_jobs=-1,
                                  class_weight='balanced')

def make_xgb():
    # Undersample 1:1 no treino já balanceia — scale_pos_weight não necessário
    return XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                         random_state=42, n_jobs=-1,
                         eval_metric='logloss', verbosity=0)

def make_svm():
    return SVC(kernel='rbf', C=1.0, probability=True,
               random_state=42, class_weight='balanced')

def make_lr():
    return LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced')

def make_nb(): return GaussianNB()
def make_knn(k): return KNeighborsClassifier(n_neighbors=k)
def make_mdc(): return MinimumDistanceClassifier()

MODELS = {
    'RF':   make_rf,
    'XGB':  make_xgb,
    'SVM':  make_svm,
    'LR':   make_lr,
    'NB':   make_nb,
    'kNN3': lambda: make_knn(3),
    'kNN5': lambda: make_knn(5),
    'kNN7': lambda: make_knn(7),
    'MDC':  make_mdc,
}
print(f'Modelos: {list(MODELS.keys())}')

Modelos: ['RF', 'XGB', 'SVM', 'LR', 'NB', 'kNN3', 'kNN5', 'kNN7', 'MDC']


## 4. Carregamento dos dados por paciente

In [5]:
def load_patient_features(fp, zscore_by_ctx=ZSCORE_BY_CTX):
    raw  = np.load(fp, allow_pickle=True)
    meta = json.loads(str(raw['meta']))
    X_pre         = raw['X_pre'].copy().astype(np.float32)
    X_inter       = raw['X_inter'].copy().astype(np.float32)
    t_pre         = raw['t_pre'].copy()
    t_inter       = raw['t_inter'].copy()
    ctx_ids_pre   = raw['ctx_ids_pre'].copy()
    ctx_ids_inter = raw['ctx_ids_inter'].copy()
    if zscore_by_ctx:
        for ctx_id in np.unique(ctx_ids_pre):
            mask = ctx_ids_pre == ctx_id
            if mask.sum() < 2: continue
            mu = X_pre[mask].mean(axis=0, keepdims=True)
            sd = X_pre[mask].std(axis=0,  keepdims=True) + 1e-10
            X_pre[mask] = (X_pre[mask] - mu) / sd
        for ctx_id in np.unique(ctx_ids_inter):
            mask = ctx_ids_inter == ctx_id
            if mask.sum() < 2: continue
            mu = X_inter[mask].mean(axis=0, keepdims=True)
            sd = X_inter[mask].std(axis=0,  keepdims=True) + 1e-10
            X_inter[mask] = (X_inter[mask] - mu) / sd
    return {'X_pre':X_pre, 'X_inter':X_inter, 't_pre':t_pre, 't_inter':t_inter,
            'ctx_ids_pre':ctx_ids_pre, 'ctx_ids_inter':ctx_ids_inter, 'meta':meta}

# N_Ctx_Total canônico do manifesto
N_CTX_CANONICAL = {}
if MANIFEST.exists():
    with open(MANIFEST, encoding='utf-8') as f: manifest = json.load(f)
    for c in manifest.get('contexts', []):
        if c.get('npz_saved', False):
            key = (c['dataset'], c['paciente'])
            N_CTX_CANONICAL[key] = N_CTX_CANONICAL.get(key, 0) + 1
    print(f'Manifesto: {len(N_CTX_CANONICAL)} pacientes, {sum(N_CTX_CANONICAL.values())} contextos')
else:
    print('AVISO: pipeline_manifest.json não encontrado')

# PRE_SEC individual (W5)
PRE_SEC_BY_PATIENT = {}
if PRE_EST.exists():
    with open(PRE_EST, encoding='utf-8') as f: pre_est_raw = json.load(f)
    for entry in pre_est_raw:
        key = (entry['dataset'], entry['paciente'])
        if entry.get('pre_sec') is not None:
            PRE_SEC_BY_PATIENT[key] = float(entry['pre_sec'])
    print(f'PRE_SEC carregado para {len(PRE_SEC_BY_PATIENT)} pacientes.')
else:
    print('AVISO: preictal_estimate.json não encontrado — W5 usará PRESEC_FALLBACK_S')

# Carrega todos os pacientes
PATIENTS_DATA = {}
for fp in sorted(FEAT_DIR.glob('*.npz')):
    d = load_patient_features(fp)
    ds, pat = d['meta']['dataset'], d['meta']['paciente']
    PATIENTS_DATA[(ds, pat)] = d
print(f'{len(PATIENTS_DATA)} pacientes carregados.')
for ds in sorted(set(k[0] for k in PATIENTS_DATA)):
    n = sum(1 for k in PATIENTS_DATA if k[0] == ds)
    print(f'  {ds}: {n} pacientes')

Manifesto: 26 pacientes, 127 contextos
PRE_SEC carregado para 18 pacientes.
26 pacientes carregados.
  CHBMIT: 7 pacientes
  Mendeley: 5 pacientes
  SeizeIT2: 7 pacientes
  Siena: 7 pacientes


## 5. Funções centrais — filtro, LOSO, seleção de janela

**Opção B (FP/h):** o interictal do teste usa o segmento **completo** do contexto,
não filtrado por W. Isso torna o FP/h clinicamente válido e comparável entre janelas.
O treino continua usando apenas W segundos de inter (balanceado 1:1).

In [6]:
def filter_window(d, W_s, level_cols=None):
    t_pre, t_inter = d['t_pre'], d['t_inter']
    X_pre, X_inter = d['X_pre'], d['X_inter']
    cid_pre, cid_inter = d['ctx_ids_pre'], d['ctx_ids_inter']
    mask_p = t_pre   >= -W_s
    mask_i = t_inter >= -W_s
    Xp, Xi = X_pre[mask_p], X_inter[mask_i]
    cp, ci = cid_pre[mask_p], cid_inter[mask_i]
    if level_cols is not None:
        n_cols = min(level_cols, Xp.shape[1], Xi.shape[1])
        Xp, Xi = Xp[:, :n_cols], Xi[:, :n_cols]
    return Xp, Xi, cp, ci

def get_level_cols(d, level):
    slices = d['meta']['level_slices']
    return slices[level] if level in slices else max(slices.values())

def _train_eval_single(Xp, Xi, cp, ci, train_contexts, test_context, model_name,
                       Xi_full=None, ci_full=None):
    tr_p_mask = np.isin(cp, train_contexts)
    tr_i_mask = np.isin(ci, train_contexts)
    Xp_tr, Xi_tr = Xp[tr_p_mask], Xi[tr_i_mask]
    _base = {'fold_ctx': int(test_context), 'auc': np.nan, 'f1': np.nan,
             'sensitivity': np.nan, 'specificity': np.nan, 'fp_h': np.nan,
             'n_train': 0, 'n_test': 0}
    if len(Xp_tr) == 0 or len(Xi_tr) == 0:
        return {**_base, 'erro': 'treino_vazio'}
    te_p_mask = cp == test_context
    Xp_te = Xp[te_p_mask]
    # Opção B: interictal completo no teste
    if Xi_full is not None and ci_full is not None:
        Xi_te = Xi_full[ci_full == test_context]
    else:
        Xi_te = Xi[ci == test_context]
    if len(Xp_te) == 0 or len(Xi_te) == 0:
        return {**_base, 'n_train': len(Xp_tr)+len(Xi_tr), 'erro': 'teste_vazio'}
    X_test = np.vstack([Xp_te, Xi_te])
    y_test = np.concatenate([np.ones(len(Xp_te)), np.zeros(len(Xi_te))])
    X_test = np.nan_to_num(X_test, nan=0.0, posinf=1e6, neginf=-1e6)
    n_min = min(len(Xp_tr), len(Xi_tr))
    seed_metrics = []
    for seed in SEEDS:
        rng   = np.random.RandomState(seed)
        idx_p = rng.choice(len(Xp_tr), n_min, replace=False)
        idx_i = rng.choice(len(Xi_tr), n_min, replace=False)
        X_train = np.vstack([Xp_tr[idx_p], Xi_tr[idx_i]])
        y_train = np.concatenate([np.ones(n_min), np.zeros(n_min)])
        X_train = np.nan_to_num(X_train, nan=0.0, posinf=1e6, neginf=-1e6)
        scaler  = StandardScaler()
        Xtr_s   = scaler.fit_transform(X_train)
        Xte_s   = scaler.transform(X_test)
        Xtr_s   = np.clip(np.nan_to_num(Xtr_s, nan=0., posinf=0., neginf=0.), -50, 50)
        Xte_s   = np.clip(np.nan_to_num(Xte_s, nan=0., posinf=0., neginf=0.), -50, 50)
        if N_FEATURES_SELECT is not None and Xtr_s.shape[1] > N_FEATURES_SELECT:
            sel   = SelectKBest(mutual_info_classif, k=N_FEATURES_SELECT)
            Xtr_s = sel.fit_transform(Xtr_s, y_train)
            Xte_s = sel.transform(Xte_s)
        try:
            clf    = MODELS[model_name]()
            clf.fit(Xtr_s, y_train)
            y_pred = clf.predict(Xte_s)
            y_scr  = (clf.predict_proba(Xte_s)[:,1]
                      if hasattr(clf,'predict_proba') else y_pred.astype(float))
            auc_ = roc_auc_score(y_test, y_scr) if len(set(y_test))>1 else np.nan
            f1_  = f1_score(y_test, y_pred, zero_division=0)
            tn, fp_, fn, tp_ = confusion_matrix(y_test, y_pred, labels=[0,1]).ravel()
            sens_ = tp_/(tp_+fn) if (tp_+fn)>0 else np.nan
            spec_ = tn/(tn+fp_) if (tn+fp_)>0 else np.nan
            dur_h = (len(Xi_te) * 15.0) / 3600.0
            fph_  = fp_/dur_h if dur_h>0 else np.nan
            seed_metrics.append({'auc':auc_,'f1':f1_,'sensitivity':sens_,
                                 'specificity':spec_,'fp_h':fph_})
        except Exception as e:
            seed_metrics.append({'auc':np.nan,'f1':np.nan,'sensitivity':np.nan,
                                 'specificity':np.nan,'fp_h':np.nan,'erro_seed':str(e)})
    def _mean(key):
        vals = [m[key] for m in seed_metrics if not np.isnan(m.get(key, np.nan))]
        return float(np.mean(vals)) if vals else np.nan
    return {'fold_ctx':int(test_context),
            'auc':_mean('auc'), 'f1':_mean('f1'),
            'sensitivity':_mean('sensitivity'), 'specificity':_mean('specificity'),
            'fp_h':_mean('fp_h'), 'n_train':len(X_train), 'n_test':len(X_test),
            'n_seeds_ok':sum(1 for m in seed_metrics if not np.isnan(m.get('auc',np.nan)))}

def _run_loso_generic(Xp, Xi, cp, ci, contexts, model_name, Xi_full=None, ci_full=None):
    results = []
    for held_out in contexts:
        train_ctxs = [c for c in contexts if c != held_out]
        if not train_ctxs: continue
        r = _train_eval_single(Xp, Xi, cp, ci, train_ctxs, held_out, model_name,
                               Xi_full=Xi_full, ci_full=ci_full)
        if r is not None: results.append(r)
    return results

def run_loso_patient(d, W_s, level=None, model_name='RF'):
    level_cols = get_level_cols(d, level) if level else None
    Xp, Xi, cp, ci = filter_window(d, W_s, level_cols)
    if len(Xp)==0 or len(Xi)==0: return []
    contexts = sorted(set(cp) & set(ci))  # interseção — Correção 1
    if len(contexts) < 2: return []
    Xi_full = d['X_inter']
    ci_full = d['ctx_ids_inter']
    if level_cols is not None:
        Xi_full = Xi_full[:, :min(level_cols, Xi_full.shape[1])]
    return _run_loso_generic(Xp, Xi, cp, ci, contexts, model_name,
                             Xi_full=Xi_full, ci_full=ci_full)

def select_window_nested(d, candidate_windows, level=None, model_name='RF'):
    _Xi_full_raw = d['X_inter']
    _ci_full_raw = d['ctx_ids_inter']
    level_cols   = get_level_cols(d, level) if level else None
    cached = {}
    for wlabel, W_s in candidate_windows.items():
        Xp, Xi, cp, ci = filter_window(d, W_s, level_cols)
        cached[wlabel] = (Xp, Xi, cp, ci, W_s)
    # Correção 1: UNIÃO das interseções
    valid_per_w = [set(v[2]) & set(v[3]) for v in cached.values()]
    all_contexts = sorted(set().union(*valid_per_w))
    if len(all_contexts) < 2: return []
    # Correção 3: 2 contextos → sem aninhamento
    if len(all_contexts) == 2:
        best_wlabel = max(candidate_windows, key=lambda k: candidate_windows[k])
        Xp, Xi, cp, ci, W_s = cached[best_wlabel]
        _lc = get_level_cols(d, level) if level else None
        _Xf = _Xi_full_raw[:, :min(_lc, _Xi_full_raw.shape[1])] if _lc else _Xi_full_raw
        results = []
        for r in _run_loso_generic(Xp, Xi, cp, ci, all_contexts, model_name,
                                   Xi_full=_Xf, ci_full=_ci_full_raw):
            r['chosen_window'] = best_wlabel
            r['chosen_W_s']    = W_s
            r['fallback_2ctx'] = True
            results.append(r)
        return results
    # >= 3 contextos: LOSO aninhado
    results = []
    for held_out in all_contexts:
        inner = [c for c in all_contexts if c != held_out]
        if len(inner) < 2: continue
        inner_auc = {}
        for wlabel, (Xp, Xi, cp, ci, W_s) in cached.items():
            valid_inner = sorted(set(cp) & set(ci) & set(inner))
            if len(valid_inner) < 2:
                inner_auc[wlabel] = -np.inf; continue
            inner_res = _run_loso_generic(Xp, Xi, cp, ci, valid_inner, model_name)
            aucs = [r['auc'] for r in inner_res if not np.isnan(r['auc'])]
            inner_auc[wlabel] = np.mean(aucs) if aucs else -np.inf
        if all(v == -np.inf for v in inner_auc.values()): continue
        best_wlabel = max(inner_auc, key=inner_auc.get)
        Xp, Xi, cp, ci, W_s = cached[best_wlabel]
        _lc = get_level_cols(d, level) if level else None
        _Xf = _Xi_full_raw[:, :min(_lc, _Xi_full_raw.shape[1])] if _lc else _Xi_full_raw
        r = _train_eval_single(Xp, Xi, cp, ci, inner, held_out, model_name,
                               Xi_full=_Xf, ci_full=_ci_full_raw)
        if r is not None:
            r['chosen_window'] = best_wlabel
            r['chosen_W_s']    = W_s
            results.append(r)
    return results

print('Funções LOSO definidas.')

Funções LOSO definidas.


## 5b. Teste comparativo em chb06 — Config A vs Config B

Valida mudanças de hiperparâmetros antes de rodar o pipeline completo.

- **Config A (atual):** `N_FEATURES_SELECT=None`, sem `min_samples_leaf`, `n_estimators=200`
- **Config B (referência):** `N_FEATURES_SELECT=70`, `min_samples_leaf=5`, `n_estimators=300`


In [7]:
_ds_test, _pat_test = 'CHBMIT', 'chb06'
_key_test = (_ds_test, _pat_test)

if _key_test not in PATIENTS_DATA:
    print(f'AVISO: chb06 não encontrado. Disponíveis: {[p for _,p in PATIENTS_DATA.keys()]}')
else:
    _d_test = PATIENTS_DATA[_key_test]
    _cand   = dict(WINDOWS_FIXED)
    _cand['W5'] = PRE_SEC_BY_PATIENT.get(_key_test, PRESEC_FALLBACK_S)

    print('='*60)
    print('CONFIG A (atual) — N_FEATURES_SELECT=None, sem min_samples_leaf')
    print('='*60)
    _res_a = select_window_nested(_d_test, _cand, level=None, model_name='RF')
    _df_a  = pd.DataFrame(_res_a)
    print(_df_a[['fold_ctx','chosen_window','auc','f1','sensitivity','specificity','fp_h']].to_string(index=False))
    print(f'AUC medio: {_df_a["auc"].mean():.4f} +/- {_df_a["auc"].std():.4f}')

CONFIG A (atual) — N_FEATURES_SELECT=None, sem min_samples_leaf
 fold_ctx chosen_window      auc       f1  sensitivity  specificity       fp_h
        1            W4 0.972543 0.692670     0.911111     0.880801  28.607679
        2            W5 0.687872 0.209719     0.877778     0.414357 140.554257
        3            W4 0.481241 0.245865     0.870707     0.138564 206.744574
        4            W5 0.646293 0.192592     0.814815     0.400668 143.839733
        5            W5 0.518670 0.165859     0.829630     0.262771 176.934891
        6            W5 0.937748 0.304083     0.970370     0.602003  95.519199
        7            W5 0.863260 0.164305     1.000000     0.082805 220.126878
AUC medio: 0.7297 +/- 0.1979


In [8]:
if _key_test in PATIENTS_DATA:
    from sklearn.ensemble import RandomForestClassifier as _RFC
    _orig_make_rf    = make_rf
    _orig_nfeat      = N_FEATURES_SELECT
    _g = globals()
    def _make_rf_b():
        return _RFC(n_estimators=300, max_depth=12, min_samples_leaf=5,
                    random_state=42, n_jobs=-1, class_weight='balanced')
    _g['make_rf'] = _make_rf_b
    _g['N_FEATURES_SELECT'] = 70

    print('='*60)
    print('CONFIG B (referência) — N_FEATURES_SELECT=70, min_samples_leaf=5, n_estimators=300')
    print('='*60)
    _cand_b = dict(WINDOWS_FIXED)
    _cand_b['W5'] = PRE_SEC_BY_PATIENT.get(_key_test, PRESEC_FALLBACK_S)
    _res_b = select_window_nested(PATIENTS_DATA[_key_test], _cand_b, level=None, model_name='RF')
    _df_b  = pd.DataFrame(_res_b)
    print(_df_b[['fold_ctx','chosen_window','auc','f1','sensitivity','specificity','fp_h']].to_string(index=False))
    print(f'AUC medio: {_df_b["auc"].mean():.4f} +/- {_df_b["auc"].std():.4f}')

    print()
    print('COMPARATIVO — Config A vs Config B')
    _comp = pd.DataFrame({
        'Metrica':  ['AUC','F1','Sens','Spec'],
        'Config A': [_df_a['auc'].mean(), _df_a['f1'].mean(),
                     _df_a['sensitivity'].mean(), _df_a['specificity'].mean()],
        'Config B': [_df_b['auc'].mean(), _df_b['f1'].mean(),
                     _df_b['sensitivity'].mean(), _df_b['specificity'].mean()],
    })
    _comp['Delta(A-B)'] = (_comp['Config A'] - _comp['Config B']).round(4)
    print(_comp.round(4).to_string(index=False))
    print('Delta positivo = Config A melhor')

    _g['make_rf']           = _orig_make_rf
    _g['N_FEATURES_SELECT'] = _orig_nfeat
    print()
    print('Configuração original (Config A) restaurada.')

CONFIG B (referência) — N_FEATURES_SELECT=70, min_samples_leaf=5, n_estimators=300
 fold_ctx chosen_window      auc       f1  sensitivity  specificity       fp_h
        1            W4 0.976530 0.743028     0.911111     0.910518  21.475793
        2            W4 0.897958 0.400329     0.953535     0.535559 111.465776
        3            W4 0.493324 0.247752     0.905051     0.107179 214.277129
        4            W4 0.828693 0.471137     0.705051     0.787312  51.045075
        5            W4 0.831494 0.383588     0.842424     0.578297 101.208681
        6            W3 0.867486 0.355560     0.878481     0.595993  96.961603
        7            W4 0.847952 0.263045     0.939394     0.139900 206.424040
AUC medio: 0.8205 +/- 0.1530

COMPARATIVO — Config A vs Config B
Metrica  Config A  Config B  Delta(A-B)
    AUC    0.7297    0.8205     -0.0908
     F1    0.2822    0.4092     -0.1270
   Sens    0.8963    0.8764      0.0199
   Spec    0.3974    0.5221     -0.1247
Delta positivo = Con

## 6. Etapa 1 — Qual janela W performa melhor?

Janela escolhida **por fold** via LOSO interno. Fixo: nível máximo por paciente, RF.

**Nota sobre W5:** estimado sobre o mesmo sinal — interpretar com cautela.

In [9]:
rows_s1 = []
for (ds, pat), d in tqdm(sorted(PATIENTS_DATA.items()), desc='Etapa 1 — janelas'):
    cand = dict(WINDOWS_FIXED)
    cand['W5'] = PRE_SEC_BY_PATIENT.get((ds, pat), PRESEC_FALLBACK_S)
    for r in select_window_nested(d, cand, level=None, model_name='RF'):
        rows_s1.append({'dataset': ds, 'paciente': pat, **r})
df_s1 = pd.DataFrame(rows_s1)
df_s1.to_csv(OUT_DIR / 'stage1_windows.csv', index=False)
print(f'Etapa 1: {len(df_s1)} folds -> stage1_windows.csv')

Etapa 1 — janelas: 100%|██████████| 26/26 [3:28:09<00:00, 480.36s/it]   

Etapa 1: 127 folds -> stage1_windows.csv


In [10]:
agg_s1 = (df_s1.groupby('chosen_window')
               .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                    f1_mean=('f1','mean'), sens_mean=('sensitivity','mean'),
                    spec_mean=('specificity','mean'), fp_h_mean=('fp_h','mean'),
                    n_folds=('fold_ctx','count'))
               .reset_index().sort_values('auc_mean', ascending=False))
print('RANKING DE JANELAS (Etapa 1)')
print('='*100)
print(f'  {"Janela":<8}{"AUC":<16}{"F1":<8}{"Sens.":<8}{"Espec.":<8}{"FP/h":<8}{"n"}')
for _, row in agg_s1.iterrows():
    print(f'  {row["chosen_window"]:<8}'
          f'{row["auc_mean"]:.3f}+/-{row["auc_std"]:.3f}   '
          f'{row["f1_mean"]:<8.3f}{row["sens_mean"]:<8.3f}'
          f'{row["spec_mean"]:<8.3f}{row["fp_h_mean"]:<8.2f}{int(row["n_folds"])}')
print(f'Global: AUC={df_s1["auc"].mean():.3f}+/-{df_s1["auc"].std():.3f}  '
      f'F1={df_s1["f1"].mean():.3f}  FP/h={df_s1["fp_h"].mean():.2f}')

PATIENT_WINDOW = df_s1.groupby(['dataset','paciente'])['chosen_W_s'].median().to_dict()
print()
print('JANELA CONSENSO POR PACIENTE')
print(f'  {"Dataset":<10} {"Paciente":<12} {"W(min)":<8} {"Moda":<6} folds/ctx_real')
for (ds, pat), W_s in sorted(PATIENT_WINDOW.items()):
    sub_p = df_s1[(df_s1.dataset==ds)&(df_s1.paciente==pat)]
    moda  = sub_p['chosen_window'].value_counts().idxmax()
    n_f   = sub_p['fold_ctx'].nunique()
    n_r   = N_CTX_CANONICAL.get((ds, pat), '?')
    fb    = 'fallback_2ctx' in sub_p.columns and sub_p.get('fallback_2ctx', pd.Series([False])).any()
    flag  = ' [2ctx]' if fb else ''
    print(f'  {ds:<10} {pat:<12} {W_s/60:>5.1f}   {moda:<6} {n_f}/{n_r}{flag}')

RANKING DE JANELAS (Etapa 1)
  Janela  AUC             F1      Sens.   Espec.  FP/h    n
  W5      0.735+/-0.208   0.330   0.788   0.505   118.82  27
  W1      0.643+/-0.252   0.265   0.734   0.462   129.23  32
  W2      0.624+/-0.317   0.350   0.662   0.514   116.56  25
  W4      0.577+/-0.284   0.443   0.724   0.446   133.08  35
  W3      0.428+/-0.261   0.218   0.366   0.626   89.88   8
Global: AUC=0.627+/-0.274  F1=0.342  FP/h=123.10

JANELA CONSENSO POR PACIENTE
  Dataset    Paciente     W(min)   Moda   folds/ctx_real
  CHBMIT     chb03         15.6   W5     3/3
  CHBMIT     chb05         28.4   W5     3/3
  CHBMIT     chb06         13.7   W5     7/7
  CHBMIT     chb07         10.2   W1     3/3
  CHBMIT     chb08         20.0   W4     5/5
  CHBMIT     chb10         15.0   W2     6/6
  CHBMIT     chb14         10.0   W1     5/5
  Mendeley   p10           25.0   W4     2/2 [2ctx]
  Mendeley   p11           25.0   W4     2/2 [2ctx]
  Mendeley   p12           25.0   W4     2/2 [2ctx]


## 7. Etapa 2 — Qual nível de canais performa melhor?

SeizeIT2 participa apenas com R0 (2 canais nativos).
O melhor nível é reportado global e por dataset.

In [11]:
LEVELS = ['R0','R1','R2','R3','R4','R5']
rows_s2 = []
for (ds, pat), d in tqdm(sorted(PATIENTS_DATA.items()), desc='Etapa 2 — níveis'):
    W_s = PATIENT_WINDOW.get((ds, pat))
    if W_s is None: continue
    for level in LEVELS:
        for r in run_loso_patient(d, W_s, level=level, model_name='RF'):
            rows_s2.append({'dataset':ds,'paciente':pat,'level':level,'W_s':W_s,**r})
df_s2 = pd.DataFrame(rows_s2)
df_s2.to_csv(OUT_DIR / 'stage2_levels.csv', index=False)
print(f'Etapa 2: {len(df_s2)} folds -> stage2_levels.csv')

Etapa 2 — níveis: 100%|██████████| 26/26 [36:14<00:00, 83.63s/it] 

Etapa 2: 762 folds -> stage2_levels.csv


In [12]:
agg_s2 = (df_s2.groupby('level')
               .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                    f1_mean=('f1','mean'), sens_mean=('sensitivity','mean'),
                    spec_mean=('specificity','mean'), fp_h_mean=('fp_h','mean'),
                    n_folds=('fold_ctx','count'))
               .reset_index())
agg_s2['level'] = pd.Categorical(agg_s2['level'], categories=LEVELS, ordered=True)
agg_s2 = agg_s2.sort_values('level')
print('RANKING DE NÍVEIS (Etapa 2)')
print('='*100)
print(f'  {"Nível":<8}{"AUC":<16}{"F1":<8}{"Sens.":<8}{"Espec.":<8}{"FP/h":<8}{"n"}')
for _, row in agg_s2.iterrows():
    print(f'  {row["level"]:<8}'
          f'{row["auc_mean"]:.3f}+/-{row["auc_std"]:.3f}   '
          f'{row["f1_mean"]:<8.3f}{row["sens_mean"]:<8.3f}'
          f'{row["spec_mean"]:<8.3f}{row["fp_h_mean"]:<8.2f}{int(row["n_folds"])}')
BEST_LEVEL = agg_s2.loc[agg_s2['auc_mean'].idxmax(), 'level']
print(f'Melhor nível global: {BEST_LEVEL}')
print('Melhor nível por dataset:')
for ds in sorted(df_s2['dataset'].unique()):
    sub = df_s2[df_s2['dataset']==ds].groupby('level')['auc'].mean()
    sub.index = pd.CategoricalIndex(sub.index, categories=LEVELS, ordered=True)
    bd = sub.sort_index().idxmax()
    print(f'  {ds:<12} -> {bd}  (AUC={sub[bd]:.3f})')

RANKING DE NÍVEIS (Etapa 2)
  Nível   AUC             F1      Sens.   Espec.  FP/h    n
  R0      0.626+/-0.276   0.338   0.679   0.493   121.80  127
  R1      0.642+/-0.270   0.352   0.710   0.493   121.78  127
  R2      0.647+/-0.271   0.355   0.732   0.486   123.35  127
  R3      0.650+/-0.271   0.354   0.730   0.491   122.25  127
  R4      0.649+/-0.272   0.357   0.731   0.489   122.74  127
  R5      0.651+/-0.268   0.358   0.729   0.494   121.45  127
Melhor nível global: R5
Melhor nível por dataset:
  CHBMIT       -> R2  (AUC=0.697)
  Mendeley     -> R1  (AUC=0.517)
  SeizeIT2     -> R2  (AUC=0.671)
  Siena        -> R1  (AUC=0.640)


## 8. Etapa 3 — Qual modelo performa melhor?

RF é o modelo principal. Os demais são análise secundária.
SeizeIT2 usa R0; demais datasets usam BEST_LEVEL.

In [13]:
rows_s3 = []
for (ds, pat), d in tqdm(sorted(PATIENTS_DATA.items()), desc='Etapa 3 — modelos'):
    W_s = PATIENT_WINDOW.get((ds, pat))
    if W_s is None: continue
    level = 'R0' if ds == 'SeizeIT2' else BEST_LEVEL
    for model_name in MODELS:
        for r in run_loso_patient(d, W_s, level=level, model_name=model_name):
            rows_s3.append({'dataset':ds,'paciente':pat,'model':model_name,'W_s':W_s,**r})
df_s3 = pd.DataFrame(rows_s3)
df_s3.to_csv(OUT_DIR / 'stage3_models.csv', index=False)
print(f'Etapa 3: {len(df_s3)} folds -> stage3_models.csv')

Etapa 3 — modelos:   0%|          | 0/26 [00:00<?, ?it/s]  File "c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 556, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 1038, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                 

Etapa 3: 1143 folds -> stage3_models.csv


In [14]:
ALL_MODELS = ['RF','XGB','SVM','LR','NB','kNN3','kNN5','kNN7','MDC']
agg_s3 = (df_s3.groupby(['dataset','model'])
               .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                    f1_mean=('f1','mean'), sens_mean=('sensitivity','mean'),
                    spec_mean=('specificity','mean'), fp_h_mean=('fp_h','mean'),
                    n_folds=('fold_ctx','count'))
               .reset_index())
print('AUC POR DATASET E MODELO')
print('='*112)
header = f'  {"Dataset":<12} ' + ' '.join(f'{m:<10}' for m in ALL_MODELS) + '  Melhor'
print(header); print('  ' + '-'*(len(header)-2))
for ds in sorted(agg_s3['dataset'].unique()):
    sub  = agg_s3[agg_s3['dataset']==ds].set_index('model')
    vals = {m: sub.loc[m,'auc_mean'] if m in sub.index else float('nan') for m in ALL_MODELS}
    best = max(vals, key=lambda k: vals[k] if not (isinstance(vals[k],float) and vals[k]!=vals[k]) else -1)
    row  = f'  {ds:<12} ' + ' '.join(f'{vals[m]:<10.3f}' if vals[m]==vals[m] else f'{"n/a":<10}' for m in ALL_MODELS)
    print(row + f'  {best}')

gbl_full = (df_s3.groupby('model')
                 .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                      f1_mean=('f1','mean'), sens_mean=('sensitivity','mean'),
                      spec_mean=('specificity','mean'), fp_h_mean=('fp_h','mean'),
                      n_folds=('fold_ctx','count'))
                 .reindex(ALL_MODELS).sort_values('auc_mean', ascending=False))
print()
print('MÉTRICAS GLOBAIS POR MODELO')
print(f'  {"Modelo":<8}{"AUC":<16}{"F1":<8}{"Sens.":<8}{"Espec.":<8}{"FP/h":<8}{"n"}')
for m, row in gbl_full.iterrows():
    print(f'  {m:<8}{row["auc_mean"]:.3f}+/-{row["auc_std"]:.3f}   '
          f'{row["f1_mean"]:<8.3f}{row["sens_mean"]:<8.3f}'
          f'{row["spec_mean"]:<8.3f}{row["fp_h_mean"]:<8.2f}{int(row["n_folds"])}')
gbl = df_s3.groupby('model')['auc'].mean()
best_global = gbl.dropna().idxmax()
print(f'Melhor modelo global: {best_global}  (AUC={gbl[best_global]:.3f})')

AUC POR DATASET E MODELO
  Dataset      RF         XGB        SVM        LR         NB         kNN3       kNN5       kNN7       MDC         Melhor
  -----------------------------------------------------------------------------------------------------------------------
  CHBMIT       0.691      0.681      0.611      0.502      0.543      0.627      0.637      0.638      0.518       RF
  Mendeley     0.467      0.463      0.521      0.507      0.516      0.503      0.508      0.514      0.563       MDC
  SeizeIT2     0.671      0.679      0.612      0.520      0.529      0.588      0.597      0.599      0.525       XGB
  Siena        0.629      0.599      0.617      0.528      0.499      0.603      0.612      0.614      0.486       RF

MÉTRICAS GLOBAIS POR MODELO
  Modelo  AUC             F1      Sens.   Espec.  FP/h    n
  RF      0.651+/-0.268   0.358   0.729   0.494   121.45  127
  XGB     0.646+/-0.288   0.369   0.713   0.491   122.07  127
  SVM     0.605+/-0.166   0.311   0.722   0.

## 9. Sanidade — N_Ctx por paciente

In [15]:
print('VERIFICAÇÃO N_Ctx')
for df, nome in [(df_s1,'Etapa 1'),(df_s2,'Etapa 2'),(df_s3,'Etapa 3')]:
    if df.empty: continue
    n_ctx = df.groupby(['dataset','paciente'])['fold_ctx'].nunique()
    prob  = n_ctx[n_ctx == 1]
    if len(prob):
        print(f'  AVISO {nome}: {len(prob)} paciente(s) com N_Ctx=1 (interictal curto para W)')
        for (ds, pat), _ in prob.items():
            print(f'    {ds}/{pat}  N_real={N_CTX_CANONICAL.get((ds,pat),"?")}')
    else:
        print(f'  OK {nome}: sem pacientes com N_Ctx=1')

VERIFICAÇÃO N_Ctx
  OK Etapa 1: sem pacientes com N_Ctx=1
  OK Etapa 2: sem pacientes com N_Ctx=1
  OK Etapa 3: sem pacientes com N_Ctx=1


## 10. Resumo final e exportação

In [16]:
summary = {
    'versao': 'NB4-v5',
    'janelas_min': {k: v//60 for k, v in WINDOWS_FIXED.items()},
    'presec_fallback_min': PRESEC_FALLBACK_S / 60,
    'w_default_min': W_DEFAULT_S // 60,
    'correcoes': ['intersecao-contextos','dict-nan','fallback-2ctx','n-ctx-manifesto'],
    'melhorias': ['zscore-por-ctx', f'undersample-{N_SEEDS}-seeds',
                  'opcao-b-inter-completo', 'rf-sem-min-samples-leaf', 'selectkbest-off'],
    'best_level': str(BEST_LEVEL),
    'best_model_global': str(best_global),
    'best_model_auc_global': float(gbl[best_global]),
    'patient_windows_min': {f'{k[0]}/{k[1]}': round(v/60,1) for k,v in PATIENT_WINDOW.items()},
    'stage1_ranking': agg_s1.to_dict(orient='records'),
    'stage2_ranking': agg_s2.to_dict(orient='records'),
    'stage3_ranking_ds_model': agg_s3.to_dict(orient='records'),
    'stage3_global': gbl_full.reset_index().to_dict(orient='records'),
}
SUMMARY_PATH = OUT_DIR / 'nb4_summary.json'
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=str)
print('RESUMO FINAL')
print(f'  Nível ótimo : {BEST_LEVEL}')
print(f'  Modelo ótimo: {best_global}  (AUC={gbl[best_global]:.3f})')
print(f'  Resultados: {OUT_DIR}/')
try:
    from google.colab import files; files.download(str(SUMMARY_PATH))
except ImportError:
    print(f'  Sumário: {SUMMARY_PATH.resolve()}')

RESUMO FINAL
  Nível ótimo : R5
  Modelo ótimo: RF  (AUC=0.651)
  Resultados: data\results/
  Sumário: D:\TCC\data\results\nb4_summary.json


## 11. Tabelas detalhadas por paciente

In [17]:
from IPython.display import display, HTML

def show_stage_by_dataset(csv_name, group_col, title):
    fp = OUT_DIR / csv_name
    if not fp.exists():
        print(f'{csv_name} não encontrado.'); return
    df = pd.read_csv(fp)
    if N_CTX_CANONICAL:
        df['N_Ctx_Total'] = df.apply(
            lambda r: N_CTX_CANONICAL.get((r['dataset'], r['paciente']), None), axis=1)
    else:
        n_ctx_csv = df.groupby(['dataset','paciente'])['fold_ctx'].nunique().rename('N_Ctx_Total')
        df = df.merge(n_ctx_csv.reset_index(), on=['dataset','paciente'])
    agg = (df.groupby(['dataset','paciente',group_col])
             .agg(AUC=('auc','mean'), AUC_std=('auc','std'),
                  F1=('f1','mean'), Sens=('sensitivity','mean'),
                  Espec=('specificity','mean'), FP_h=('fp_h','mean'),
                  N_Ctx=('fold_ctx','nunique'), N_Ctx_Total=('N_Ctx_Total','first'))
             .reset_index()
             .rename(columns={group_col: group_col.capitalize(), 'paciente': 'Paciente'}))
    display(HTML(f'<h3>{title}</h3>'))
    for ds in sorted(agg['dataset'].unique()):
        sub = (agg[agg['dataset']==ds].drop(columns='dataset')
                  .sort_values(['Paciente', group_col.capitalize()])
                  .reset_index(drop=True))
        cols = ['Paciente','N_Ctx_Total',group_col.capitalize(),
                'N_Ctx','AUC','AUC_std','F1','Sens','Espec','FP_h']
        sub  = sub[[c for c in cols if c in sub.columns]]
        display(HTML(f'<h4>{ds}</h4>'))
        styled = (sub.style
                     .background_gradient(subset=['AUC'], cmap='RdYlGn', vmin=0.4, vmax=1.0)
                     .format({'AUC':'{:.3f}','AUC_std':'{:.3f}','F1':'{:.3f}',
                              'Sens':'{:.3f}','Espec':'{:.3f}','FP_h':'{:.2f}'}))
        display(styled)

show_stage_by_dataset('stage1_windows.csv', 'chosen_window',
    'ETAPA 1 — Desempenho por paciente (janela via LOSO aninhado)')
show_stage_by_dataset('stage2_levels.csv', 'level',
    'ETAPA 2 — Desempenho por paciente (nível de canais)')
show_stage_by_dataset('stage3_models.csv', 'model',
    'ETAPA 3 — Desempenho por paciente (modelo)')

,Paciente,N_Ctx_Total,Chosen_window,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,chb03,3,W4,1,0.900,nan,0.477,1.000,0.093,217.71
1,chb03,3,W5,2,0.744,0.330,0.473,0.924,0.431,136.67
2,chb05,3,W2,1,0.604,nan,0.227,0.173,0.914,20.69
3,chb05,3,W5,2,0.956,0.045,0.779,0.939,0.734,63.77
4,chb06,7,W4,2,0.727,0.347,0.469,0.891,0.510,117.68
5,chb06,7,W5,5,0.731,0.169,0.207,0.899,0.353,155.39
6,chb07,3,W1,1,0.621,nan,0.125,1.000,0.088,218.84
7,chb07,3,W2,1,0.727,nan,0.180,0.953,0.148,204.50
8,chb07,3,W5,1,0.723,nan,0.191,0.980,0.443,133.66
9,chb08,5,W1,1,0.749,nan,0.300,0.979,0.258,178.14


,Paciente,N_Ctx_Total,Chosen_window,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,p10,2,W4,2,0.109,0.087,0.436,0.930,0.030,232.85
1,p11,2,W4,2,0.665,0.111,0.430,0.438,0.700,72.08
2,p12,2,W4,2,0.607,0.094,0.339,0.487,0.541,110.24
3,p13,2,W4,2,0.398,0.178,0.321,0.537,0.348,156.44
4,p15,3,W3,3,0.528,0.187,0.223,0.247,0.690,74.36


,Paciente,N_Ctx_Total,Chosen_window,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,sub-002,5,W1,1,0.739,nan,0.233,0.574,0.781,52.65
1,sub-002,5,W3,1,0.328,nan,0.067,0.048,0.950,12.02
2,sub-002,5,W5,3,0.738,0.270,0.252,0.863,0.422,138.82
3,sub-034,5,W2,1,0.473,nan,0.162,0.566,0.465,128.29
4,sub-034,5,W4,3,0.528,0.324,0.229,0.487,0.479,125.04
5,sub-034,5,W5,1,0.519,nan,0.322,0.533,0.484,123.87
6,sub-035,8,W1,4,0.758,0.205,0.254,0.974,0.430,136.92
7,sub-035,8,W2,1,0.952,nan,0.312,1.000,0.563,104.81
8,sub-035,8,W4,2,0.211,0.098,0.228,0.831,0.034,231.95
9,sub-035,8,W5,1,0.704,nan,0.179,1.000,0.023,234.58


,Paciente,N_Ctx_Total,Chosen_window,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,PN01,2,W4,2,0.458,0.106,0.264,0.883,0.068,223.69
1,PN05,2,W4,2,0.804,0.277,0.724,0.999,0.496,121.02
2,PN06,4,W1,1,0.550,nan,0.289,0.720,0.263,176.81
3,PN06,4,W3,1,0.049,nan,0.064,0.037,0.919,19.42
4,PN06,4,W4,2,0.939,0.060,0.836,0.888,0.865,32.41
5,PN09,3,W4,2,0.519,0.003,0.488,0.872,0.293,169.79
6,PN09,3,W5,1,0.417,nan,0.163,0.335,0.593,97.64
7,PN10,5,W2,1,0.411,nan,0.280,0.763,0.155,202.81
8,PN10,5,W4,1,0.652,nan,0.383,0.790,0.162,201.15
9,PN10,5,W5,3,0.640,0.115,0.268,0.648,0.574,102.32


,Paciente,N_Ctx_Total,Level,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,chb03,3,R0,3,0.374,0.122,0.285,0.583,0.376,149.76
1,chb03,3,R1,3,0.565,0.175,0.363,0.805,0.312,165.15
2,chb03,3,R2,3,0.722,0.276,0.414,0.882,0.363,152.77
3,chb03,3,R3,3,0.723,0.264,0.396,0.876,0.327,161.47
4,chb03,3,R4,3,0.732,0.273,0.417,0.901,0.348,156.38
5,chb03,3,R5,3,0.763,0.235,0.450,0.919,0.397,144.80
6,chb05,3,R0,3,0.620,0.185,0.493,0.605,0.584,99.75
7,chb05,3,R1,3,0.738,0.207,0.468,0.533,0.838,38.76
8,chb05,3,R2,3,0.831,0.197,0.550,0.642,0.833,40.17
9,chb05,3,R3,3,0.875,0.170,0.594,0.665,0.856,34.61


...

,Paciente,N_Ctx_Total,Level,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,PN01,2,R0,2,0.380,0.053,0.261,0.850,0.068,223.58
1,PN01,2,R1,2,0.535,0.125,0.272,0.880,0.113,212.83
2,PN01,2,R2,2,0.599,0.207,0.275,0.919,0.070,223.23
3,PN01,2,R3,2,0.584,0.192,0.276,0.938,0.054,227.02
4,PN01,2,R4,2,0.548,0.175,0.276,0.920,0.070,223.20
5,PN01,2,R5,2,0.458,0.106,0.264,0.883,0.068,223.69
6,PN05,2,R0,2,0.810,0.217,0.640,0.935,0.455,130.78
7,PN05,2,R1,2,0.806,0.157,0.560,0.964,0.300,167.92
8,PN05,2,R2,2,0.779,0.239,0.622,0.927,0.426,137.78
9,PN05,2,R3,2,0.829,0.195,0.635,0.958,0.425,137.90


,Paciente,N_Ctx_Total,Model,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,chb03,3,LR,3,0.367,0.059,0.271,0.581,0.304,167.03
1,chb03,3,MDC,3,0.413,0.137,0.259,0.457,0.470,127.20
2,chb03,3,NB,3,0.469,0.185,0.296,0.629,0.393,145.61
3,chb03,3,RF,3,0.763,0.235,0.450,0.919,0.397,144.80
4,chb03,3,SVM,3,0.542,0.133,0.351,0.796,0.291,170.04
5,chb03,3,XGB,3,0.536,0.379,0.395,0.919,0.251,179.75
6,chb03,3,kNN3,3,0.544,0.077,0.343,0.731,0.361,153.31
7,chb03,3,kNN5,3,0.545,0.078,0.327,0.688,0.357,154.31
8,chb03,3,kNN7,3,0.533,0.094,0.325,0.688,0.351,155.65
9,chb05,3,LR,3,0.513,0.010,0.444,0.681,0.344,157.39


,Paciente,N_Ctx_Total,Model,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,p10,2,LR,2,0.480,0.081,0.461,0.798,0.296,168.99
1,p10,2,MDC,2,0.578,0.204,0.460,0.788,0.313,164.98
2,p10,2,NB,2,0.557,0.080,0.448,0.949,0.053,227.39
3,p10,2,RF,2,0.109,0.087,0.436,0.930,0.030,232.85
4,p10,2,SVM,2,0.515,0.173,0.475,0.874,0.235,183.55
5,p10,2,XGB,2,0.179,0.250,0.204,0.414,0.034,231.95
6,p10,2,kNN3,2,0.495,0.036,0.432,0.793,0.218,187.65
7,p10,2,kNN5,2,0.495,0.045,0.441,0.803,0.235,183.62
8,p10,2,kNN7,2,0.506,0.042,0.431,0.798,0.208,190.19
9,p11,2,LR,2,0.485,0.119,0.492,0.788,0.332,160.27


,Paciente,N_Ctx_Total,Model,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,sub-002,5,LR,5,0.619,0.174,0.192,0.792,0.376,149.71
1,sub-002,5,MDC,5,0.594,0.155,0.211,0.683,0.539,110.66
2,sub-002,5,NB,5,0.576,0.203,0.128,0.229,0.768,55.68
3,sub-002,5,RF,5,0.749,0.192,0.214,0.647,0.590,98.50
4,sub-002,5,SVM,5,0.649,0.220,0.199,0.775,0.400,143.93
5,sub-002,5,XGB,5,0.716,0.203,0.210,0.608,0.610,93.55
6,sub-002,5,kNN3,5,0.589,0.102,0.177,0.729,0.381,148.57
7,sub-002,5,kNN5,5,0.599,0.101,0.174,0.717,0.395,145.29
8,sub-002,5,kNN7,5,0.606,0.106,0.184,0.717,0.421,139.04
9,sub-034,5,LR,5,0.523,0.046,0.338,0.642,0.397,144.62


,Paciente,N_Ctx_Total,Model,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,PN01,2,LR,2,0.453,0.019,0.234,0.714,0.143,205.59
1,PN01,2,MDC,2,0.446,0.004,0.226,0.580,0.302,167.52
2,PN01,2,NB,2,0.543,0.028,0.292,0.850,0.220,187.29
3,PN01,2,RF,2,0.458,0.106,0.264,0.883,0.068,223.69
4,PN01,2,SVM,2,0.429,0.051,0.214,0.606,0.184,195.82
5,PN01,2,XGB,2,0.268,0.042,0.221,0.732,0.053,227.18
6,PN01,2,kNN3,2,0.461,0.024,0.228,0.514,0.414,140.63
7,PN01,2,kNN5,2,0.463,0.023,0.223,0.495,0.419,139.39
8,PN01,2,kNN7,2,0.457,0.013,0.223,0.500,0.414,140.61
9,PN05,2,LR,2,0.522,0.016,0.457,0.680,0.425,138.00
